# 5. Pipeline de Machine Learning

En este notebook se muestra una versión integrada del flujo de trabajo usando Pipeline para organizar el preprocesamiento y el modelo.

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score
from sklearn.ensemble import GradientBoostingClassifier

In [ ]:
df = pd.read_csv("data/train.csv")

# Selección de variables útiles
features = ["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked"]
X = df[features]
y = df["Survived"]

X.head()

## Preprocesamiento

Se imputan valores faltantes, se escalan variables numéricas y se codifican variables categóricas.

In [ ]:
variables_numericas = ["Pclass", "Age", "SibSp", "Parch", "Fare"]
variables_categoricas = ["Sex", "Embarked"]

preprocesamiento_numerico = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

preprocesamiento_categorico = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocesador = ColumnTransformer(
    transformers=[
        ("num", preprocesamiento_numerico, variables_numericas),
        ("cat", preprocesamiento_categorico, variables_categoricas)
    ]
)

## Creación del pipeline

El pipeline une el preprocesamiento y el modelo para evitar errores y mantener el proceso ordenado.

In [ ]:
pipeline = Pipeline(steps=[
    ("preprocesador", preprocesador),
    ("modelo", GradientBoostingClassifier(random_state=42))
])

parametros = {
    "modelo__n_estimators": [10, 100],
    "modelo__max_depth": [1, 2, 3],
    "modelo__learning_rate": [0.05, 0.1, 0.2]
}

## Entrenamiento y evaluación

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=parametros,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

y_pred = grid_search.predict(X_test)
precision = accuracy_score(y_test, y_pred)

print("Mejores hiperparámetros:", grid_search.best_params_)
print(f"Precisión del modelo: {precision:.2f}")

## Conclusión final

El pipeline permite construir un flujo de Machine Learning más limpio y reproducible. En este proyecto, las variables más relevantes para predecir la supervivencia fueron el género, la clase del boleto, la edad y la tarifa pagada.